In [4]:
from tifffile import imread, imwrite

data_path = "/Users/sam/Library/CloudStorage/OneDrive-UniversityofCambridge/Synapse_localisation/Andre/code/sam-for-mask/data/s5_Parker.tif"
volume = imread(data_path)


In [1]:
import numpy as np
from scipy import ndimage
from skimage import filters, morphology, feature
from typing import Tuple, Optional
import gc


def adaptive_em_masking(
    volume: np.ndarray,
    sigma_range: Tuple[float, float] = (0.5, 2.0),
    n_scales: int = 3,
    threshold_factor: float = 1.2,
    chunk_size: Optional[Tuple[int, int, int]] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Memory-efficient adaptive multi-scale EM data masking.
    
    Parameters:
    -----------
    volume : ndarray
        3D input EM volume
    sigma_range : tuple
        Range of Gaussian sigma values to use (min, max)
    n_scales : int
        Number of scales to process
    threshold_factor : float
        Factor to multiply standard deviation for thresholding
    chunk_size : tuple, optional
        Size of chunks for processing (z, y, x). If None, processes whole volume.
        
    Returns:
    --------
    mask : ndarray
        Binary mask of the same shape as input
    confidence : ndarray
        Confidence map (0-1) indicating reliability of masking
    """
    
    # Determine chunk size if not provided
    if chunk_size is None:
        # Default to chunks of roughly 64MB
        chunk_size = (
            min(64, volume.shape[0]),
            min(512, volume.shape[1]),
            min(512, volume.shape[2])
        )
    
    # Initialize output arrays
    mask = np.zeros_like(volume, dtype=bool)
    confidence = np.zeros_like(volume, dtype=np.float32)
    
    # Calculate number of chunks
    nz = int(np.ceil(volume.shape[0] / chunk_size[0]))
    ny = int(np.ceil(volume.shape[1] / chunk_size[1]))
    nx = int(np.ceil(volume.shape[2] / chunk_size[2]))
    
    # Generate sigma values
    sigmas = np.geomspace(sigma_range[0], sigma_range[1], n_scales)
    
    # Process volume in chunks
    for iz in range(nz):
        z_start = iz * chunk_size[0]
        z_end = min((iz + 1) * chunk_size[0], volume.shape[0])
        
        for iy in range(ny):
            y_start = iy * chunk_size[1]
            y_end = min((iy + 1) * chunk_size[1], volume.shape[1])
            
            for ix in range(nx):
                x_start = ix * chunk_size[2]
                x_end = min((ix + 1) * chunk_size[2], volume.shape[2])
                
                # Extract chunk with padding
                pad = int(max(sigmas) * 3)  # 3 sigma padding
                
                z_pad_start = max(0, z_start - pad)
                z_pad_end = min(volume.shape[0], z_end + pad)
                y_pad_start = max(0, y_start - pad)
                y_pad_end = min(volume.shape[1], y_end + pad)
                x_pad_start = max(0, x_start - pad)
                x_pad_end = min(volume.shape[2], x_end + pad)
                
                chunk = volume[z_pad_start:z_pad_end,
                             y_pad_start:y_pad_end,
                             x_pad_start:x_pad_end]
                
                # Normalize chunk
                chunk_norm = (chunk - chunk.min()) / (chunk.max() - chunk.min())
                
                # Process chunk
                dog_responses = np.zeros_like(chunk_norm)
                
                for i in range(len(sigmas)-1):
                    sigma1, sigma2 = sigmas[i], sigmas[i+1]
                    g1 = ndimage.gaussian_filter(chunk_norm, sigma1)
                    g2 = ndimage.gaussian_filter(chunk_norm, sigma2)
                    dog_responses += np.abs(g2 - g1)
                    
                    # Free memory
                    del g1, g2
                    gc.collect()
                
                dog_responses /= n_scales - 1
                
                # Compute adaptive threshold
                local_mean = ndimage.uniform_filter(dog_responses, size=5)
                local_std = np.sqrt(
                    ndimage.uniform_filter(dog_responses**2, size=5) 
                    - local_mean**2
                )
                
                threshold = local_mean + threshold_factor * local_std
                chunk_mask = dog_responses > threshold
                
                # Compute confidence
                chunk_confidence = np.clip(
                    (dog_responses - threshold) / (threshold * 0.5),
                    0, 1
                )
                
                # Remove padding
                z_offset = z_pad_start - z_start
                y_offset = y_pad_start - y_start
                z_slice = slice(pad + z_offset, pad + z_offset + z_end - z_start)
                y_slice = slice(pad + y_offset, pad + y_offset + y_end - y_start)
                x_offset = x_pad_start - x_start
                x_slice = slice(pad + x_offset, pad + x_offset + x_end - x_start)
                
                # Store results
                mask[z_start:z_end,
                     y_start:y_end,
                     x_start:x_end] = chunk_mask[z_slice, y_slice, x_slice]
                
                confidence[z_start:z_end,
                          y_start:y_end,
                          x_start:x_end] = chunk_confidence[z_slice, y_slice, x_slice]
                
                # Clean up
                del chunk, chunk_norm, dog_responses, local_mean, local_std
                gc.collect()
    
    # Final cleanup using morphological operations
    for z in range(0, volume.shape[0], chunk_size[0]):
        z_end = min(z + chunk_size[0], volume.shape[0])
        mask[z:z_end] = morphology.remove_small_objects(
            mask[z:z_end], min_size=64
        )
        mask[z:z_end] = morphology.remove_small_holes(
            mask[z:z_end], area_threshold=64
        )
    
    return mask, confidence

In [2]:

# For a 300MB volume
chunk_size = (32, 256, 256)  # Adjust based on your available RAM
mask, confidence = adaptive_em_masking(
    volume,
    chunk_size=chunk_size,
    sigma_range=(0.5, 2.0),
    n_scales=3
)

print(confidence)
# write the mask file
imwrite("./DOG_mask_s5_Parker.tiff", mask)


/var/folders/7y/0myqmy515cnfk4c9_tcy30l00000gn/T/ipykernel_9793/1810315956.py:116: RuntimeWarning: invalid value encountered in divide
  (dog_responses - threshold) / (threshold * 0.5),


[[[       nan 0.         0.         ... 0.         0.         0.        ]
  [       nan 0.         0.         ... 0.19705719 0.         0.        ]
  [       nan 0.         0.         ... 0.         0.         0.        ]
  ...
  [       nan 0.         0.         ... 0.         0.         0.        ]
  [       nan 0.         0.         ... 0.         0.         0.01872811]
  [       nan 0.         0.         ... 0.         0.         0.        ]]

 [[       nan 0.         0.         ... 0.15132457 0.1002231  0.        ]
  [       nan 0.         0.         ... 0.5696255  0.359998   0.        ]
  [       nan 0.         0.         ... 0.         0.         0.        ]
  ...
  [       nan 0.         0.         ... 0.         0.         0.        ]
  [       nan 0.         0.         ... 0.         0.         0.        ]
  [       nan 0.         0.         ... 0.         0.         0.        ]]

 [[       nan 0.         0.         ... 0.         0.         0.        ]
  [       nan 0.      

In [10]:
import numpy as np
from scipy import ndimage
from skimage import filters, feature, morphology
from typing import Tuple, Optional

def em_texture_masking(
    volume: np.ndarray,
    block_size: int = 31,
    texture_sigma: float = 2.0,
    edge_sigma: float = 1.0,
    chunk_size: Optional[Tuple[int, int, int]] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    EM-specific masking using local texture and structure analysis.
    
    Parameters:
    -----------
    volume : ndarray
        3D input EM volume
    block_size : int
        Size of block for local statistics (odd number)
    texture_sigma : float
        Sigma for texture filtering
    edge_sigma : float
        Sigma for edge detection
    chunk_size : tuple, optional
        Size of chunks for processing (z, y, x)
        
    Returns:
    --------
    mask : ndarray
        Binary mask of the same shape as input
    confidence : ndarray
        Confidence map (0-1) indicating reliability of masking
    """
    
    def process_chunk(chunk: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # Normalize chunk
        chunk_norm = (chunk - np.percentile(chunk, 1)) / (
            np.percentile(chunk, 99) - np.percentile(chunk, 1)
        )
        chunk_norm = np.clip(chunk_norm, 0, 1)
        
        print(f"footprint: {morphology.disk(block_size//2)}")
        # 1. Local texture analysis
        local_var = ndimage.variance(chunk_norm, index=morphology.disk(block_size//2)) # footprint=morphology.disk(block_size//2)
        texture_mask = local_var > filters.threshold_otsu(local_var)
        
        # 2. Structure tensor analysis for oriented features
        Gx = ndimage.gaussian_filter1d(chunk_norm, texture_sigma, axis=1, order=1)
        Gy = ndimage.gaussian_filter1d(chunk_norm, texture_sigma, axis=0, order=1)
        
        Gxx = Gx * Gx
        Gyy = Gy * Gy
        Gxy = Gx * Gy
        
        # Compute coherence
        coherence = np.sqrt((Gxx - Gyy)**2 + 4*Gxy**2) / (Gxx + Gyy + 1e-6)
        coherence = ndimage.gaussian_filter(coherence, edge_sigma)
        
        # 3. Edge detection using Scharr operator
        edges = np.zeros_like(chunk_norm)
        for i in range(chunk_norm.shape[0]):
            edges[i] = filters.scharr(chunk_norm[i])
        edges = ndimage.gaussian_filter(edges, edge_sigma)
        
        # 4. Combine evidence
        combined_evidence = (
            0.4 * texture_mask +
            0.3 * (coherence > filters.threshold_otsu(coherence)) +
            0.3 * (edges > filters.threshold_otsu(edges))
        )
        
        # 5. Final mask with adaptive thresholding
        final_mask = combined_evidence > 0.5
        
        # 6. Clean up
        final_mask = morphology.remove_small_objects(final_mask, min_size=100)
        final_mask = morphology.remove_small_holes(final_mask, area_threshold=100)
        
        # Compute confidence based on evidence strength
        confidence = ndimage.gaussian_filter(combined_evidence, sigma=1.0)
        
        return final_mask, confidence
    
    # Handle chunking
    if chunk_size is None:
        chunk_size = (
            min(64, volume.shape[0]),
            min(512, volume.shape[1]),
            min(512, volume.shape[2])
        )
    
    # Initialize output arrays
    mask = np.zeros_like(volume, dtype=bool)
    confidence = np.zeros_like(volume, dtype=np.float32)
    
    # Process chunks
    for z in range(0, volume.shape[0], chunk_size[0]):
        z_end = min(z + chunk_size[0], volume.shape[0])
        for y in range(0, volume.shape[1], chunk_size[1]):
            y_end = min(y + chunk_size[1], volume.shape[1])
            for x in range(0, volume.shape[2], chunk_size[2]):
                x_end = min(x + chunk_size[2], volume.shape[2])
                
                chunk = volume[z:z_end, y:y_end, x:x_end]
                chunk_mask, chunk_conf = process_chunk(chunk)
                
                mask[z:z_end, y:y_end, x:x_end] = chunk_mask
                confidence[z:z_end, y:y_end, x:x_end] = chunk_conf
    
    return mask, confidence

def interactive_refinement(
    volume: np.ndarray,
    mask: np.ndarray,
    roi_coords: Tuple[slice, slice, slice],
    feature_type: str = 'texture'
) -> np.ndarray:
    """
    Refine mask based on local texture/structure in ROI
    
    Parameters:
    -----------
    volume : ndarray
        Original volume
    mask : ndarray
        Current mask
    roi_coords : tuple of slices
        ROI coordinates
    feature_type : str
        'texture' or 'structure' based refinement
        
    Returns:
    --------
    refined_mask : ndarray
    """
    roi = volume[roi_coords]
    roi_mask = mask[roi_coords]
    
    if feature_type == 'texture':
        # Extract texture signature from ROI
        local_var = ndimage.variance(roi, size=15)
        texture_sig = np.median(local_var[roi_mask])
        texture_std = np.std(local_var[roi_mask])
        
        # Update mask based on similar textures
        full_var = ndimage.variance(volume, size=15)
        similarity = np.abs(full_var - texture_sig) < (2 * texture_std)
        
        return mask | similarity
    
    else:  # structure
        # Use structure tensor for orientation-based refinement
        # Implementation similar to main algorithm
       pass
    
    # return refined_mask

In [11]:
# Basic usage
mask, confidence = em_texture_masking(
    volume,
    block_size=31,  # Adjust based on feature size
    texture_sigma=2.0,
    edge_sigma=1.0
)

footprint: [[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0]
 [0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0]
 [0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0]
 [0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [1 1 1 1 1 1 

In [12]:
# write the mask file
imwrite("./TEXTURE_mask_s5_Parker_v2.tiff", mask)


In [20]:
import numpy as np
from skimage import morphology, segmentation, feature
from scipy import ndimage
import numpy.typing as npt
from typing import Tuple, Optional, List
import cv2


def create_filled_outline(
    mask: npt.NDArray,
    outline_width: int = 2,
    inside_value: int = 255,
    outside_value: int = 0
) -> npt.NDArray:
    """
    Create a filled mask based on the outline of an input mask.
    
    Parameters:
    -----------
    mask : NDArray
        Input binary mask
    outline_width : int
        Width of the outline to consider
    inside_value : int
        Value to fill inside the outline
    outside_value : int
        Value for regions outside the outline
        
    Returns:
    --------
    filled_mask : NDArray
        New mask with filled outline
    """
    # Initialize output with outside value
    filled_mask = np.full_like(mask, outside_value, dtype=np.uint8)
    
    # Process each slice for 3D volumes
    for z in range(mask.shape[0]):
        # Get current slice
        slice_mask = mask[z].astype(np.uint8)
        
        # Find contours
        contours, _ = cv2.findContours(
            slice_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )
        
        # Fill contours
        if contours:
            cv2.drawContours(
                filled_mask[z],
                contours,
                -1,
                inside_value,
                -1  # Fill interior
            )
    
    return filled_mask

# First create filled outline
filled_mask = create_filled_outline(mask)

# write the new mask
imwrite("./TEXTURE_mask_filled.tiff", filled_mask)



In [29]:
import numpy as np
from scipy import ndimage
from skimage import filters, feature, morphology, measure
from typing import Tuple, Optional
import cv2

def filter_artifacts(
    mask: np.ndarray,
    min_size: int = 10000,
    max_eccentricity: float = 0.85,
    center_region_size: float = 0.15
) -> np.ndarray:
    """
    Filter out artifacts in 3D mask based on size, shape, and position
    
    Parameters:
    -----------
    mask : ndarray
        3D binary mask
    min_size : int
        Minimum region size in pixels per slice
    max_eccentricity : float
        Maximum eccentricity (0-1) for circular filtering
    center_region_size : float
        Size of center region to check (as fraction of image size)
    """
    clean_mask = np.zeros_like(mask, dtype=bool)
    
    # Process each slice
    for z in range(mask.shape[0]):
        # Get current slice
        slice_mask = mask[z]
        
        # Get slice center and dimensions
        center_y, center_x = np.array(slice_mask.shape) // 2
        h, w = slice_mask.shape
        center_region_h = int(h * center_region_size)
        center_region_w = int(w * center_region_size)
        
        # Label connected components in slice
        labels = measure.label(slice_mask)
        regions = measure.regionprops(labels)
        
        for region in regions:
            # Skip small regions
            if region.area < min_size:
                continue
                
            # Skip highly eccentric (non-circular) regions
            if region.eccentricity > max_eccentricity:
                continue
            
            # Check if region is entirely within center region
            cy, cx = region.centroid
            in_center = (
                abs(cy - center_y) < center_region_h/2 and
                abs(cx - center_x) < center_region_w/2 and
                region.area < min_size * 2  # Smaller threshold for center objects
            )
            
            if not in_center:
                clean_mask[z][labels == region.label] = True
    
    # Optional: Add 3D connectivity check
    if mask.shape[0] > 1:  # Only if we have multiple slices
        # Label 3D connected components
        labels_3d = measure.label(clean_mask, connectivity=1)
        regions_3d = measure.regionprops(labels_3d)
        
        # Filter small 3D regions
        min_volume = min_size * 2  # Adjust this multiplier as needed
        for region in regions_3d:
            if region.area < min_volume:
                clean_mask[labels_3d == region.label] = False
    
    return clean_mask


def em_texture_masking(
    volume: np.ndarray,
    block_size: int = 31,
    texture_sigma: float = 2.0,
    edge_sigma: float = 1.0,
    min_region_size: int = 10000,
    chunk_size: Optional[Tuple[int, int, int]] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    EM-specific masking with artifact removal
    
    Parameters:
    -----------
    volume : ndarray
        3D input EM volume
    block_size : int
        Size of block for local statistics
    texture_sigma : float
        Sigma for texture filtering
    edge_sigma : float
        Sigma for edge detection
    min_region_size : int
        Minimum size for valid regions
    chunk_size : tuple, optional
        Size of chunks for processing
    """
    def process_chunk(chunk: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # Normalize chunk using robust statistics
        p1, p99 = np.percentile(chunk, (1, 99))
        chunk_norm = np.clip((chunk - p1) / (p99 - p1), 0, 1)
        
        # 1. Enhanced texture analysis
        local_var = ndimage.variance(chunk_norm, 
                                   index=morphology.disk(block_size//2))
        texture_mask = local_var > filters.threshold_otsu(local_var)
        
        # 2. Structure tensor analysis
        Gx = ndimage.gaussian_filter1d(chunk_norm, texture_sigma, axis=1, order=1)
        Gy = ndimage.gaussian_filter1d(chunk_norm, texture_sigma, axis=0, order=1)
        
        Gxx = Gx * Gx
        Gyy = Gy * Gy
        Gxy = Gx * Gy
        
        coherence = np.sqrt((Gxx - Gyy)**2 + 4*Gxy**2) / (Gxx + Gyy + 1e-6)
        coherence = ndimage.gaussian_filter(coherence, edge_sigma)
        
        # 3. Edge detection
        edges = filters.scharr(chunk_norm)
        edges = ndimage.gaussian_filter(edges, edge_sigma)
        
        # 4. Combine evidence
        combined_evidence = (
            0.4 * texture_mask +
            0.3 * (coherence > filters.threshold_otsu(coherence)) +
            0.3 * (edges > filters.threshold_otsu(edges))
        )
        
        # 5. Initial mask
        initial_mask = combined_evidence > 0.5
        
        # # 6. Filter artifacts
        # clean_mask = filter_artifacts(
        #     initial_mask,
        #     min_size=min_region_size,
        #     max_eccentricity=0.85,
        #     center_region_size=0.15
        # )
        clean_mask = initial_mask
        
        # 7. Compute confidence
        confidence = ndimage.gaussian_filter(combined_evidence, sigma=1.0)
        confidence[~clean_mask] = 0  # Zero confidence in filtered regions
        
        return clean_mask, confidence
    
    # Handle chunking
    if chunk_size is None:
        chunk_size = (
            min(64, volume.shape[0]),
            min(512, volume.shape[1]),
            min(512, volume.shape[2])
        )
    
    # Initialize output arrays
    mask = np.zeros_like(volume, dtype=bool)
    confidence = np.zeros_like(volume, dtype=np.float32)
    
    # Process chunks
    for z in range(0, volume.shape[0], chunk_size[0]):
        z_end = min(z + chunk_size[0], volume.shape[0])
        for y in range(0, volume.shape[1], chunk_size[1]):
            y_end = min(y + chunk_size[1], volume.shape[1])
            for x in range(0, volume.shape[2], chunk_size[2]):
                x_end = min(x + chunk_size[2], volume.shape[2])
                
                chunk = volume[z:z_end, y:y_end, x:x_end]
                chunk_mask, chunk_conf = process_chunk(chunk)
                
                mask[z:z_end, y:y_end, x:x_end] = chunk_mask
                confidence[z:z_end, y:y_end, x:x_end] = chunk_conf
    
    return mask, confidence

# Basic usage with artifact filtering
mask, confidence = em_texture_masking(
    volume,
    block_size=64,
    texture_sigma=1.0,
    edge_sigma=0.5,
    min_region_size=250  # Adjust based on your image size
)

# clean_mask = filter_artifacts(
#             mask,
#             min_size=250,
#             max_eccentricity=0.85,
#             center_region_size=0.15
#         )

# create filled outline
filled_mask = create_filled_outline(mask)

# write the mask file
imwrite("./TEXTURE_mask_s5_Parker_v3.tiff", filled_mask)


In [ ]:
import napari


class InteractiveMaskRefiner:
    def __init__(
        self,
        volume: npt.NDArray,
        mask: npt.NDArray,
        chunk_size: int = 5
    ):
        """
        Interactive mask refinement using napari.
        
        Parameters:
        -----------
        volume : NDArray
            Original EM volume
        mask : NDArray
            Initial binary mask
        chunk_size : int
            Number of slices to process at once for 3D propagation
        """
        self.volume = volume
        self.mask = mask.astype(np.uint8)
        self.chunk_size = chunk_size
        self.viewer = None
        self.corrections = []
        
    def launch_viewer(self):
        """Launch napari viewer with required layers"""
        self.viewer = napari.Viewer()
        
        # Add volume layer
        self.viewer.add_image(
            self.volume,
            name='EM Volume',
            contrast_limits=[self.volume.min(), self.volume.max()]
        )
        
        # Add mask layer
        self.mask_layer = self.viewer.add_labels(
            self.mask,
            name='Mask'
        )
        
        # Add points layer for corrections
        self.points_layer = self.viewer.add_points(
            name='Correction Points',
            size=5,
            face_color='red'
        )
        
        # Add buttons
        self.viewer.window.add_dock_widget(
            self._create_button_widget(),
            area='right'
        )
        
        return self.viewer
    
    def _create_button_widget(self):
        """Create widget with correction buttons"""
        from qtpy.QtWidgets import QWidget, QVBoxLayout, QPushButton
        
        widget = QWidget()
        layout = QVBoxLayout()
        
        # Add correction button
        correct_btn = QPushButton('Apply Correction')
        correct_btn.clicked.connect(self._apply_correction)
        layout.addWidget(correct_btn)
        
        # Add propagation button
        propagate_btn = QPushButton('Propagate Corrections')
        propagate_btn.clicked.connect(self._propagate_corrections)
        layout.addWidget(propagate_btn)
        
        widget.setLayout(layout)
        return widget
    
    def _apply_correction(self):
        """Apply local correction based on points"""
        if len(self.points_layer.data) == 0:
            return
        
        current_slice = self.viewer.dims.current_step[0]
        points = self.points_layer.data
        
        # Filter points for current slice
        slice_points = points[points[:, 0] == current_slice]
        
        if len(slice_points) == 0:
            return
        
        # Create correction mask using SAM-like local refinement
        correction_mask = self._generate_local_mask(
            self.volume[current_slice],
            slice_points[:, 1:]
        )
        
        # Apply correction to current slice
        self.mask[current_slice][correction_mask] = 0
        
        # Update viewer
        self.mask_layer.data = self.mask
        self.points_layer.data = []
    
    def _generate_local_mask(
        self,
        image: npt.NDArray,
        points: npt.NDArray
    ) -> npt.NDArray:
        """
        Generate local mask around points using simple but effective method.
        Could be replaced with SAM or other advanced methods.
        """
        mask = np.zeros_like(image, dtype=bool)
        
        # Create local regions around points
        for point in points:
            y, x = point.astype(int)
            
            # Get local region
            local_region = image[
                max(0, y-32):min(image.shape[0], y+32),
                max(0, x-32):min(image.shape[1], x+32)
            ]
            
            # Simple threshold-based segmentation
            # Could be replaced with more sophisticated methods
            thresh = filters.threshold_otsu(local_region)
            local_mask = local_region > thresh
            
            # Update main mask
            mask[
                max(0, y-32):min(image.shape[0], y+32),
                max(0, x-32):min(image.shape[1], x+32)
            ] = local_mask
        
        return mask
    
    def _propagate_corrections(self):
        """Propagate corrections to neighboring slices"""
        current_slice = self.viewer.dims.current_step[0]
        
        # Define range for propagation
        start_slice = max(0, current_slice - self.chunk_size)
        end_slice = min(self.mask.shape[0], current_slice + self.chunk_size + 1)
        
        # Get current slice mask
        current_mask = self.mask[current_slice]
        
        # Propagate to neighboring slices using registration
        for z in range(start_slice, end_slice):
            if z == current_slice:
                continue
                
            # Simple propagation using registration
            # Could be replaced with more sophisticated methods
            flow = feature.register_translation(
                self.volume[current_slice],
                self.volume[z],
                upsample_factor=2
            )[0]
            
            # Apply translation
            translated_mask = ndimage.shift(
                current_mask,
                flow,
                order=0,
                mode='constant',
                cval=0
            )
            
            # Update mask with propagated corrections
            self.mask[z] = translated_mask
        
        # Update viewer
        self.mask_layer.data = self.mask
    
    def get_refined_mask(self) -> npt.NDArray:
        """Return the refined mask"""
        return self.mask

# Usage example
def refine_em_mask(
    volume: npt.NDArray,
    initial_mask: npt.NDArray
) -> npt.NDArray:
    """
    Convenience function to run the interactive refinement process.
    
    Parameters:
    -----------
    volume : NDArray
        Original EM volume
    initial_mask : NDArray
        Initial binary mask
        
    Returns:
    --------
    refined_mask : NDArray
        Final refined mask
    """
    # Create filled outline mask
    outlined_mask = create_filled_outline(initial_mask)
    
    # Initialize refiner
    refiner = InteractiveMaskRefiner(volume, outlined_mask)
    
    # Launch viewer
    viewer = refiner.launch_viewer()
    napari.run()  # Start the event loop
    
    # Return refined mask
    return refiner.get_refined_mask()